In [143]:
import regex as re

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

for i in re.finditer(PAT, "some text that i'll pre-tokenize"):
    print(i[0])

some
 text
 that
 i
'll
 pre
-
tokenize


In [144]:
from typing import BinaryIO
from cs336_basics.pretokenization_example import find_chunk_boundaries

from multiprocessing import Pool

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def pretokenize(text: bytes, special_pattern: str):
    pretoken_freq: dict[tuple[bytes, ...], int] = {}
    chunks = re.split(special_pattern, text)
    for chunk in chunks:
        for item in re.finditer(PAT, chunk):
            item = item[0].encode('utf-8')
            item = tuple(bytes([b]) for b in item)
            pretoken_freq[item] = pretoken_freq.setdefault(item, 0) + 1
    
    return pretoken_freq
    

def parallel_pretokenize(file: BinaryIO, special_tokens: list[str]):
    num_processes = 4
    boundaries = find_chunk_boundaries(file, num_processes, b"<|endoftext|>")
    
    special_pattern = "|".join(re.escape(tok) for tok in special_tokens)
    
    args = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        args.append((f.read(end - start).decode('utf-8', errors="ignore"), special_pattern))
    
    with Pool(processes=num_processes) as pool:
        results = pool.starmap(pretokenize, args)
    
    return results

        
    
with open("tests/fixtures/tinystories_sample_5M.txt", "rb") as f:
    parallel_pretokenize(f, ['<|endoftext|>'])
    

Process SpawnPoolWorker-1:
Traceback (most recent call last):
Process SpawnPoolWorker-3:
  File "/Users/kaifollmann/miniconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/kaifollmann/miniconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/kaifollmann/miniconda3/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/kaifollmann/miniconda3/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'pretokenize' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
Process SpawnPoolWorker-4:
Traceback (most recent call last):
  File "/Users/kaifollmann/miniconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
 

KeyboardInterrupt: 

In [ ]:

import regex as re

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def add_tokens(vocab: dict[int, bytes], ix: int, tokens: list[str]) -> int:
    for token in tokens:
        ix = add_token(vocab, ix, token.encode('utf-8'))
    return ix

def add_token(vocab: dict[int, bytes], ix: int, token: bytes) -> int:
    vocab[ix] = token
    return ix + 1

def train_bpe(input_path: str,
              vocab_size: int,
              special_tokens: list[str]) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    
    with open(input_path, "r") as f:
        data = f.read()
    
    vocab = {i:i.to_bytes(1, 'big') for i in range(256)}
    
    ix = len(vocab)
    ix = add_tokens(vocab, ix, special_tokens)
    
    freq_table: dict[tuple[bytes, bytes], int] = {}
    
    pretoken_freq: dict[tuple[bytes,...], int] = {}
    
    merges: list[tuple[bytes, bytes]] = []
    
    special_pattern = "|".join(re.escape(tok) for tok in special_tokens)
    
    chunks = re.split(special_pattern, data)
    
    if len(special_tokens) == 0:
        chunks = [data]
    
    for chunk in chunks:
        # Let's split:
        for item in re.finditer(PAT, chunk):
            item = item[0].encode('utf-8')
            item = tuple(bytes([b]) for b in item)
            pretoken_freq[item] = pretoken_freq.setdefault(item, 0) + 1
            
    init_phase = True
        
    while len(vocab) < vocab_size:
        # For each of these, do byte pair counting.

        if init_phase:
            init_phase = False
            for pretoken in pretoken_freq.keys():
                for i in range(len(pretoken)-1):
                    byte_pair = (pretoken[i],  pretoken[i+1])
                    freq_table[byte_pair] = freq_table.setdefault(byte_pair, 0) + pretoken_freq[pretoken]

        token = max(freq_table, key=lambda k: (freq_table[k], k))

        merges.append(token)
        ix = add_token(vocab, ix, token[0]+token[1])
        
        old_new_pretokens : list[tuple[bytes, bytes]] = []
        
        for pretoken in pretoken_freq:
            new_pretoken = ()
            i = 0
            merged = False
            
            # This here saves the pairs of indices that we are going to have to decrease/increase in count.
            index_set = set()
            while i < len(pretoken):
                if i < len(pretoken) - 1 and token == (pretoken[i], pretoken[i+1]):
                    new_pretoken = new_pretoken + (pretoken[i] + pretoken[i+1],)

                    merged = True
                    # let's insert the pairs of indices:
                    if i > 0:
                        index_set.add((i-1,i))
                    if i < len(pretoken) - 2:
                        index_set.add((i+1, i+2))
                    i += 1
                else:
                    new_pretoken = new_pretoken + pretoken[i:i+1]
                i += 1
            if merged:
                old_new_pretokens.append((pretoken, new_pretoken, index_set))
        
        for old_pretoken, new_pretoken, index_set in old_new_pretokens:
            pretoken_freq[new_pretoken] = pretoken_freq.pop(old_pretoken)
            # print(old_pretoken, new_pretoken, index_set)
            for i1, i2 in index_set:
                byte_pair = (old_pretoken[i1], old_pretoken[i2])
                freq_table[byte_pair] -= pretoken_freq[new_pretoken]
                if freq_table[byte_pair] == 0:
                    freq_table.pop(byte_pair)

            
            for i in range(len(new_pretoken)):
                a,b = token
                new_token = a+b
                if new_pretoken[i] == new_token:
                    if i > 0:
                        tmp_token = (new_pretoken[i-1], new_token)
                        freq_table[tmp_token] = freq_table.setdefault(tmp_token, 0) + pretoken_freq[new_pretoken]
                    if i < len(new_pretoken) -1:
                        tmp_token = (new_token, new_pretoken[i+1])
                        freq_table[tmp_token] = freq_table.setdefault(tmp_token, 0) + pretoken_freq[new_pretoken]
                        
                        
                        
                        
                        
                    
                # freq_table[byte_pair] = freq_table.setdefault(byte_pair, 0) + pretoken_freq[new_pretoken]
            
        freq_table.pop(token)
        
        
        # TODO: We will later substitute this with something more efficient.
        # Only for the pairs which end with a or start with b (we merged (a,b)) do we have to change the calculated statistics.
        # We maybe even know exactly by how much: In pretoken_freq, we merge letters. If we had (e,s,t) -> (e,st), then decrease (e,s) by that count and increase (e,st) by that count.
        # Invariance: The total count always stays the same.
        # If we have (e,s,t) -> (es,t), then decrease (s,t) by that count and increase (es,t) by that count.
        # Generally, if we have (a,b,c,d) -> (a,bc,d), then decrease (a,b) and (c,d) by that count, and increase (a,bc) and (bc,d) by that count.
        # We could have running account above of BEFORE and AFTER-tokens (in this case, (a,b) and (c,d)). But let's be a bit more inefficient first.
        # 
        # freq_table = {}

        
    
    return vocab, merges



vocab, merges = train_bpe('input2.txt', 256 + 1 + 6, special_tokens=["<|endoftext|>"])

In [145]:
with open("input.txt", "r") as f:
    text = f.read()

KeyboardInterrupt: 

In [ ]:
max(text.split(" "), key=len)

'"Psssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss

In [146]:
import torch
v = torch.tensor([1,2,3])

In [147]:
res = torch.zeros(2 * len(v))
res[::2] = v

In [ ]:
res

tensor([1., 0., 2., 0., 3., 0.])

In [ ]:
x = torch.linspace(start=1, end=5, steps=5)
x

tensor([1., 2., 3., 4., 5.])

In [ ]:
res.shape, x.shape

(torch.Size([6]), torch.Size([5]))

In [ ]:
result = torch.zeros_like(x)

In [ ]:
result[:-1] = x * res[1:]

In [ ]:
result

tensor([ 0.,  4.,  0., 12.,  0.,  0.])

In [ ]:
x[1:]

tensor([2., 3., 4., 5., 6.])

In [ ]:
v =  torch.linspace(1, 3, steps=2)
v

tensor([1., 3.])

In [ ]:
R = torch.zeros(2 * len(v))
R[::2] = v
R = R[:-1]
R

tensor([1., 0., 3.])

In [ ]:
x = torch.linspace(1,4,4)
R, x, x[1:]

(tensor([1., 0., 3.]), tensor([1., 2., 3., 4.]), tensor([2., 3., 4.]))

In [ ]:
result = torch.zeros_like(x)
result[..., :-1] = R * x[..., 1:]
result

tensor([ 2.,  0., 12.,  0.])

In [ ]:
result2 = torch.zeros_like(x)
result2[..., 1:] = R * x[..., :-1]
result2

tensor([0., 1., 0., 9.])

In [148]:
d = 12

In [149]:
positions = torch.tensor([4, 48, 113])

In [304]:
k = torch.arange(1, d // 2 + 1)

In [151]:
Theta = torch.pi / 2
Theta

1.5707963267948966

In [152]:
from einops import einsum

In [153]:
Thetas = 1 / Theta ** ((2*k - 2) / d)

In [154]:
seq_len = 12

In [180]:
positions

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [179]:
positions = torch.arange(start=1, end=seq_len)
angles = einsum(positions, Thetas, "pos, theta -> pos theta")
angles.shape, angles

(torch.Size([11, 6]),
 tensor([[ 1.0000,  0.9275,  0.8603,  0.7979,  0.7400,  0.6864],
         [ 2.0000,  1.8550,  1.7205,  1.5958,  1.4801,  1.3728],
         [ 3.0000,  2.7825,  2.5808,  2.3937,  2.2201,  2.0592],
         [ 4.0000,  3.7100,  3.4410,  3.1915,  2.9601,  2.7455],
         [ 5.0000,  4.6375,  4.3013,  3.9894,  3.7002,  3.4319],
         [ 6.0000,  5.5650,  5.1615,  4.7873,  4.4402,  4.1183],
         [ 7.0000,  6.4925,  6.0218,  5.5852,  5.1803,  4.8047],
         [ 8.0000,  7.4200,  6.8820,  6.3831,  5.9203,  5.4911],
         [ 9.0000,  8.3475,  7.7423,  7.1810,  6.6603,  6.1775],
         [10.0000,  9.2750,  8.6025,  7.9788,  7.4004,  6.8638],
         [11.0000, 10.2025,  9.4628,  8.7767,  8.1404,  7.5502]]))

In [181]:
cos = torch.cos(angles)
sin = torch.sin(angles)

In [182]:
cos.shape

torch.Size([11, 6])

In [183]:
from einops import repeat, einsum, rearrange

In [286]:
# position x torch matrix?
x = torch.randn(d)

In [185]:
import torch

top = torch.cat([A, -B], dim=-1)
bot = torch.cat([B,  A], dim=-1)
M   = torch.cat([top, bot], dim=-2)

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 3 but got size 1 for tensor number 1 in the list.

In [186]:
top = torch.cat([cos, -sin], dim=-1)
bot = torch.cat([sin, cos], axis=-1)
R = torch.cat([top, bot], axis=-2)
R[1]

tensor([-0.4161, -0.2804, -0.1492, -0.0250,  0.0906,  0.1967, -0.9093, -0.9599,
        -0.9888, -0.9997, -0.9959, -0.9805])

In [187]:
A = torch.tensor([[1,2],[3,4], [5,6]])
B = torch.tensor([[7,8]])

In [287]:
T = torch.stack([cos, -sin, sin, cos], dim=0)
T.shape

torch.Size([4, 11, 6])

In [288]:
S = rearrange(T, "block_dim seq_len dim_2 -> seq_len dim_2 block_dim")
S.shape

torch.Size([11, 6, 4])

In [289]:
# i = 1, d/2 = 6
S[1]

tensor([[-0.4161, -0.9093,  0.9093, -0.4161],
        [-0.2804, -0.9599,  0.9599, -0.2804],
        [-0.1492, -0.9888,  0.9888, -0.1492],
        [-0.0250, -0.9997,  0.9997, -0.0250],
        [ 0.0906, -0.9959,  0.9959,  0.0906],
        [ 0.1967, -0.9805,  0.9805,  0.1967]])

In [299]:
R = rearrange(T, "(block_dim1 block_dim2) seq_len dim_2 -> seq_len dim_2 block_dim1 block_dim2", block_dim1=2)

In [301]:
R[0]

tensor([[[ 0.5403, -0.8415],
         [ 0.8415,  0.5403]],

        [[ 0.5998, -0.8001],
         [ 0.8001,  0.5998]],

        [[ 0.6522, -0.7580],
         [ 0.7580,  0.6522]],

        [[ 0.6982, -0.7159],
         [ 0.7159,  0.6982]],

        [[ 0.7384, -0.6743],
         [ 0.6743,  0.7384]],

        [[ 0.7735, -0.6337],
         [ 0.6337,  0.7735]]])

In [302]:
R = rearrange(S, "seq_len dim_2 (r1 r2) -> seq_len dim_2 r1 r2", r1=2)
R.shape

torch.Size([11, 6, 2, 2])

In [291]:
v = rearrange(x, "(dim_2 block_dim) -> dim_2 block_dim", block_dim=2)
v.shape

torch.Size([6, 2])

In [292]:
R[0].shape, v.shape

(torch.Size([6, 2, 2]), torch.Size([6, 2]))

In [298]:
result = einsum(R[0], v, "dim_2 rows block_dim, ... dim_2 block_dim -> ... dim_2 rows")

In [294]:
result = rearrange(result, "... dim_2 block_size -> ... (dim_2 block_size)", dim_2 = 6)

In [295]:
result

tensor([ 0.3103, -0.6279, -0.3131,  0.3617,  0.0208, -2.4489, -1.5706, -0.4159,
         0.1319, -0.8931, -0.2184,  0.7397])

In [296]:
import torch

tmp = R[0]

block = torch.block_diag(*tmp)   # unpack the 6 blocks
print(block.shape)             # torch.Size([12, 12])

torch.Size([12, 12])


In [297]:
block @ x

tensor([ 0.3103, -0.6279, -0.3131,  0.3617,  0.0208, -2.4489, -1.5706, -0.4159,
         0.1319, -0.8931, -0.2184,  0.7397])

In [278]:
A @ x

tensor([ 5, 11])

In [279]:
einsum(A, x, "row col, col -> row")

tensor([ 5, 11])

In [309]:
assert torch.all(A.flatten(-1) == A)

In [321]:
A = torch.randn(2,3,4)